In [1]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import requests
from pathlib import Path

# Bounding box  44.91317,-0.63440
min_lon = -92.6006
min_lat = 29.3080
max_lon = -91.5537
max_lat = 31.4420

bbox = box(min_lon, min_lat, max_lon, max_lat)

# GBA tile URL root
BASE_URL = "https://data.source.coop/tge-labs/globalbuildingatlas-lod1"

def gba_tile_name(west, south):
    east = west + 5
    north = south + 5

    w = f"e{abs(west):03d}" if west >= 0 else f"w{abs(west):03d}"
    e = f"e{abs(east):03d}" if east >= 0 else f"w{abs(east):03d}"
    n = f"n{abs(north):02d}" if north >= 0 else f"s{abs(north):02d}"
    s = f"n{abs(south):02d}" if south >= 0 else f"s{abs(south):02d}"

    return f"{w}_{n}_{e}_{s}.parquet"

# Find intersecting 5° × 5° tiles
tiles = []

for lon in range(int(min_lon // 5) * 5, int(max_lon // 5) * 5 + 5, 5):
    for lat in range(int(min_lat // 5) * 5, int(max_lat // 5) * 5 + 5, 5):
        tiles.append(gba_tile_name(lon, lat))

print("Tiles:", tiles)

outdir = Path("gba_tiles")
outdir.mkdir(exist_ok=True)

gdfs = []

for tile in tiles:
    url = f"{BASE_URL}/{tile}"

    local_file = outdir / tile

    if not local_file.exists():
        r = requests.get(url, stream=True)
        if r.status_code == 200:
            with open(local_file, "wb") as f:
                for chunk in r.iter_content(1024*1024):
                    f.write(chunk)
        else:
            print("Missing:", tile)
            continue

    gdf = gpd.read_parquet(local_file)
    gdf = gdf[gdf.intersects(bbox)]
    gdfs.append(gdf)

if gdfs:
    result = pd.concat(gdfs)
    result.to_file("data/buildings_t63_IW2_ATLAS.geojson", driver="GeoJSON")
    print(f"Downloaded {len(result)} buildings")

Tiles: ['w095_n30_w090_n25.parquet', 'w095_n35_w090_n30.parquet']
Downloaded 440722 buildings


In [3]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject, Resampling
import numpy as np

# ------------------------------------------------
# 1. Load building polygons
# ------------------------------------------------
gdf = gpd.read_file("data/buildings_t63_IW2_ATLAS.geojson")

# ------------------------------------------------
# 2. Convert to UTM for accurate meter resolution
# ------------------------------------------------
utm_crs = gdf.estimate_utm_crs()
gdf_utm = gdf.to_crs(utm_crs)

# ------------------------------------------------
# 3. Create 1 m raster grid
# ------------------------------------------------
minx, miny, maxx, maxy = gdf_utm.total_bounds

resolution = 30  # meters

width = int((maxx - minx) / resolution)
height = int((maxy - miny) / resolution)

transform = from_origin(minx, maxy, resolution, resolution)

# ------------------------------------------------
# 4. Rasterize buildings
# ------------------------------------------------
shapes = ((geom, 1) for geom in gdf_utm.geometry)

raster = rasterize(
    shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    dtype="uint8"
)

# ------------------------------------------------
# 5. Save temporary UTM raster
# ------------------------------------------------
utm_raster = "building_30m_utm.tif"

with rasterio.open(
    utm_raster,
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="uint8",
    crs=utm_crs,
    transform=transform,
) as dst:
    dst.write(raster, 1)

# ------------------------------------------------
# 6. Reproject raster to WGS84
# ------------------------------------------------
dst_crs = "EPSG:4326"

with rasterio.open(utm_raster) as src:

    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )

    kwargs = src.meta.copy()
    kwargs.update({
        "crs": dst_crs,
        "transform": transform,
        "width": width,
        "height": height
    })

    with rasterio.open("building_30m_wgs84.tif", "w", **kwargs) as dst:

        reproject(
            source=rasterio.band(src, 1),
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )

print("Final raster saved: building_30m_wgs84.tif")

Final raster saved: building_30m_wgs84.tif
